In [1]:
import os
os.chdir("C:\\Users\\tejas\\OneDrive\\Desktop\\MscBioinfo\\Sem3\\CADD\\mini_project")

In [2]:
import pandas as pd

# Read the input CSV file
df = pd.read_csv("IDH1_testedCompounds.csv")

# Filter rows where Activity_Type is IC50
filtered_df = df[df["Activity_Type"] == "IC50"]

# Save the filtered data as a new CSV file
filtered_df.to_csv("IDH1_IC50_compounds.csv", index=False)

print("Filtering completed successfully.")
print("Number of IC50 compounds:", len(filtered_df))
print("Output file: IDH1_IC50_compounds.csv")

C:\Users\tejas\AppData\Local\Temp\ipykernel_14564\2393227387.py:4: DtypeWarning: Columns (4,5,11,15,16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("IDH1_testedCompounds.csv")


Filtering completed successfully.
Number of IC50 compounds: 13259
Output file: IDH1_IC50_compounds.csv


In [3]:
import pandas as pd
import re

# Read the IC50 CSV file
df = pd.read_csv("IDH1_IC50_compounds.csv")

# Convert BioAssay_Name to string
assay = df["BioAssay_Name"].astype(str)

# Create Mutation_Group column
df["Mutation_Group"] = ""

# --------------------------------------------------
# Group A — IDH1 R132H
# --------------------------------------------------
r132h_mask = assay.str.contains(
    r"\bR132H\b",
    case=False,
    na=False,
    regex=True
)

df.loc[r132h_mask, "Mutation_Group"] = "R132H"


# --------------------------------------------------
# Group B — Other mutant IDH1
# --------------------------------------------------

# Known IDH1 mutations
other_mutant_mask = (
    assay.str.contains(
        r"\bIDH1\b",
        case=False,
        na=False,
        regex=True
    )
    &
    assay.str.contains(
        r"\bR132C\b|\bR132G\b|\bR132S\b|\bR132L\b|\bR132P\b|\bR132V\b",
        case=False,
        na=False,
        regex=True
    )
)

# Add other mutant IDs only where they are not already R132H
df.loc[
    other_mutant_mask & ~r132h_mask,
    "Mutation_Group"
] = "Other mutant IDH1"


# --------------------------------------------------
# Group C — WT human IDH1
# --------------------------------------------------

# Look for WT / wild-type IDH1
wt_mask = (
    assay.str.contains(
        r"\bIDH1\b",
        case=False,
        na=False,
        regex=True
    )
    &
    assay.str.contains(
        r"\bWT\b|wild[- ]?type",
        case=False,
        na=False,
        regex=True
    )
)

# Only assign WT to compounds that have not already
# been classified as a mutant
df.loc[
    wt_mask & (df["Mutation_Group"] == ""),
    "Mutation_Group"
] = "WT human IDH1"


# --------------------------------------------------
# Keep only the three target groups
# --------------------------------------------------

filtered_df = df[df["Mutation_Group"] != ""]


# Save as CSV
filtered_df.to_csv(
    "IDH1_target_groups.csv",
    index=False
)


# --------------------------------------------------
# Display summary
# --------------------------------------------------

print("Filtering completed successfully.")
print()
print("Mutation Group Summary:")
print(filtered_df["Mutation_Group"].value_counts())
print()
print("Total compounds:", len(filtered_df))
print("Output file: IDH1_target_groups.csv")

Filtering completed successfully.

Mutation Group Summary:
Mutation_Group
R132H                5710
Other mutant IDH1     737
WT human IDH1         121
Name: count, dtype: int64

Total compounds: 6568
Output file: IDH1_target_groups.csv


In [4]:
import pandas as pd

# Read the input CSV
df = pd.read_csv("IDH1_target_groups.csv")

# Convert Activity_Value to numeric
df["Activity_Value"] = pd.to_numeric(
    df["Activity_Value"],
    errors="coerce"
)

# Clean Activity_Qualifier
df["Activity_Qualifier"] = (
    df["Activity_Qualifier"]
    .astype(str)
    .str.strip()
)

# Create Project_Class column
df["Project_Class"] = "Unknown"

# --------------------------------------------------
# Active: Activity_Value <= 10
# --------------------------------------------------
active_mask = (
    (df["Activity_Value"] <= 10)
    &
    (df["Activity_Qualifier"].isin(["=", "<", "<="]))
)

df.loc[active_mask, "Project_Class"] = "Active"


# --------------------------------------------------
# Moderate: 10 < Activity_Value <= 100
# --------------------------------------------------
moderate_mask = (
    (df["Activity_Value"] > 10)
    &
    (df["Activity_Value"] <= 100)
    &
    (df["Activity_Qualifier"].isin(["=", "<", "<="]))
)

df.loc[moderate_mask, "Project_Class"] = "Moderate"


# --------------------------------------------------
# Inactive: Activity_Value > 100
# --------------------------------------------------
inactive_mask = (
    (df["Activity_Value"] > 100)
    &
    (df["Activity_Qualifier"].isin(["=", ">", ">="]))
)

df.loc[inactive_mask, "Project_Class"] = "Inactive"


# --------------------------------------------------
# Keep only classified compounds
# --------------------------------------------------
filtered_df = df[df["Project_Class"] != "Unknown"]


# Save as CSV
filtered_df.to_csv(
    "IDH1_project_classified.csv",
    index=False
)


# --------------------------------------------------
# Display summary
# --------------------------------------------------
print("Classification completed successfully.")
print()
print("Project Class Summary:")
print(filtered_df["Project_Class"].value_counts())
print()
print("Total classified compounds:", len(filtered_df))
print("Output file: IDH1_project_classified.csv")

Classification completed successfully.

Project Class Summary:
Project_Class
Active      3699
Moderate      91
Inactive       3
Name: count, dtype: int64

Total classified compounds: 3793
Output file: IDH1_project_classified.csv


In [6]:
import pandas as pd

# --------------------------------------------------
# Read the CSV file
# --------------------------------------------------

file = "IDH1_project_classified.csv"
df = pd.read_csv(file)

print("=" * 70)
print("IDH1 DATASET SUMMARY")
print("=" * 70)


# --------------------------------------------------
# 1. How many are unique compounds?
# --------------------------------------------------

unique_compounds = df["Compound_CID"].nunique()

print(f"\n1. Unique compounds: {unique_compounds}")


# --------------------------------------------------
# 2. How many are R132H?
# --------------------------------------------------

r132h_count = (
    df["Mutation_Group"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("r132h")
    .sum()
)

print(f"2. R132H records: {r132h_count}")


# --------------------------------------------------
# 3. How many are other IDH1 mutants?
# --------------------------------------------------

other_mutant_count = (
    df["Mutation_Group"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("other mutant idh1")
    .sum()
)

print(f"3. Other IDH1 mutant records: {other_mutant_count}")


# --------------------------------------------------
# 4. How many are WT?
# --------------------------------------------------

wt_count = (
    df["Mutation_Group"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("wt human idh1")
    .sum()
)

print(f"4. WT human IDH1 records: {wt_count}")


# --------------------------------------------------
# 5. How many are biochemical vs cellular?
# --------------------------------------------------

# Since you confirmed there are NO biochemical assays,
# identify cellular assays from BioAssay_Name.

assay_name = df["BioAssay_Name"].astype(str)

cellular_mask = assay_name.str.contains(
    r"\bcell\b|\bcells\b|cellular|cell[- ]based|U87|HEK|HCT|tumor|cancer",
    case=False,
    na=False,
    regex=True
)

cellular_count = cellular_mask.sum()

# Everything else is considered non-cellular/other
non_cellular_count = len(df) - cellular_count

print("\n5. Assay type:")
print(f"   Cellular:              {cellular_count}")
print(f"   Non-cellular/Other:    {non_cellular_count}")


# --------------------------------------------------
# 6. How many different BioAssay IDs?
# --------------------------------------------------

unique_assays = df["BioAssay_AID"].nunique()

print(f"\n6. Different BioAssay IDs: {unique_assays}")


# --------------------------------------------------
# 7. Are all IC50 values actually in µM?
# --------------------------------------------------

print("\n7. Activity values and units:")

print("   IMPORTANT:")
print("   The provided columns do not contain an Activity_Unit column.")

print("\n   Available activity-related columns:")
print("   - Activity_Value")
print("   - Activity_Type")
print("   - Activity_Qualifier")

print("\n   Activity types found:")

print(
    df["Activity_Type"]
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
)


# --------------------------------------------------
# 8. How many records have = vs < vs > qualifiers?
# --------------------------------------------------

qualifier = (
    df["Activity_Qualifier"]
    .astype(str)
    .str.strip()
)

equal_count = (qualifier == "=").sum()
less_count = (qualifier == "<").sum()
greater_count = (qualifier == ">").sum()

less_equal_count = (qualifier == "<=").sum()
greater_equal_count = (qualifier == ">=").sum()

print("\n8. Activity Qualifiers:")
print(f"   =   : {equal_count}")
print(f"   <   : {less_count}")
print(f"   >   : {greater_count}")
print(f"   <=  : {less_equal_count}")
print(f"   >=  : {greater_equal_count}")

print("\n   All qualifiers found:")
print(qualifier.value_counts(dropna=False))


# --------------------------------------------------
# Additional summary
# --------------------------------------------------

print("\n" + "=" * 70)
print("ADDITIONAL DATASET INFORMATION")
print("=" * 70)

print(f"Total records: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print("\nActivity Type:")
print(
    df["Activity_Type"]
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
)

print("\nMutation Groups:")
print(
    df["Mutation_Group"]
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
)

print("\n" + "=" * 70)
print("ANALYSIS COMPLETED")
print("=" * 70)

IDH1 DATASET SUMMARY

1. Unique compounds: 1330
2. R132H records: 3081
3. Other IDH1 mutant records: 659
4. WT human IDH1 records: 53

5. Assay type:
   Cellular:              1222
   Non-cellular/Other:    2571

6. Different BioAssay IDs: 170

7. Activity values and units:
   IMPORTANT:
   The provided columns do not contain an Activity_Unit column.

   Available activity-related columns:
   - Activity_Value
   - Activity_Type
   - Activity_Qualifier

   Activity types found:
Activity_Type
IC50    3793
Name: count, dtype: int64

8. Activity Qualifiers:
   =   : 1379
   <   : 2411
   >   : 3
   <=  : 0
   >=  : 0

   All qualifiers found:
Activity_Qualifier
<    2411
=    1379
>       3
Name: count, dtype: int64

ADDITIONAL DATASET INFORMATION
Total records: 3793
Total columns: 34

Activity Type:
Activity_Type
IC50    3793
Name: count, dtype: int64

Mutation Groups:
Mutation_Group
R132H                3081
Other mutant IDH1     659
WT human IDH1          53
Name: count, dtype: int64

A